### Extracting GPS direction from one point

In [3]:
import pandas as pd
import math
import numpy as np

# --- Step 1: Extract direction, last valid, and stationary flag ---
def extract_direction_degrees(s, last_valid_direction):
    try:
        ew_value = int(s[-14:-10])
        ew_dir = s[-15]
        ns_value = int(s[-9:-5])
        ns_dir = s[-10]

        if ew_value == 0 and ns_value == 0:
            return last_valid_direction, last_valid_direction, True

        x = ew_value if ew_dir.upper() == 'E' else -ew_value
        y = ns_value if ns_dir.upper() == 'N' else -ns_value

        angle_rad = math.atan2(x, y)
        angle_deg = math.degrees(angle_rad) % 360

        return angle_deg, angle_deg, False
    except Exception as e:
        print(f"Error processing GPSData: {s} -> {e}")
        return None, last_valid_direction, True

# --- Step 2: Read data and apply direction + stationary ---
df = pd.read_csv('C:/thesis/climate_bike/gertjandatafiets_with_MRT_timestamp.csv')

last_valid = 90
direction_raw = []
stationary_flags = []

for s in df['GPSData']:
    angle, last_valid, stationary = extract_direction_degrees(s, last_valid)
    direction_raw.append(angle)
    stationary_flags.append(stationary)

df['direction_raw'] = direction_raw
df['stationary'] = stationary_flags

# --- Step 3: Interpolation (skipping stationary points) ---
def interpolate_direction(i, direction_series, stationary_flags):
    before = []
    after = []

    j = i - 1
    while j >= 0 and len(before) < 5:
        if not stationary_flags[j] and not np.isnan(direction_series[j]):
            before.insert(0, (j, direction_series[j]))
        j -= 1

    j = i + 1
    while j < len(direction_series) and len(after) < 5:
        if not stationary_flags[j] and not np.isnan(direction_series[j]):
            after.append((j, direction_series[j]))
        j += 1

    if len(before) == 5 and len(after) == 5:
        neighbors = before + after
        x = [idx for idx, _ in neighbors]
        y = [val for _, val in neighbors]

        sin_y = np.sin(np.deg2rad(y))
        cos_y = np.cos(np.deg2rad(y))

        coeff_sin = np.polyfit(x, sin_y, 1)
        coeff_cos = np.polyfit(x, cos_y, 1)

        sin_i = np.polyval(coeff_sin, i)
        cos_i = np.polyval(coeff_cos, i)

        angle_rad = np.arctan2(sin_i, cos_i)
        return np.degrees(angle_rad) % 360
    else:
        return direction_series[i]

df['direction'] = [
    interpolate_direction(i, df['direction_raw'], df['stationary']) for i in range(len(df))
]

# --- Step 4: Cardinal alignment check ---
def is_near_cardinal(direction, tolerance=5):
    cardinals = [0, 90, 180, 270, 360]
    return any(min(abs(direction - c), 360 - abs(direction - c)) <= tolerance for c in cardinals)

df['cardinal_aligned'] = df['direction'].apply(is_near_cardinal)

# --- Step 5: Save everything ---
df.to_csv('C:/thesis/climate_bike/gertjandatafiets_with_MRT_timestamp_dir_full.csv', index=False)

In [4]:
# calculating mrt for SOLWEIG: human body approximated as a box
import pandas as pd
import numpy as np

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('C:/thesis/climate_bike/gertjandatafiets_with_MRT_timestamp_dir_full.csv')

# -------------------------------
# Constants
# -------------------------------
sigma = 5.67e-8                # Stefan-Boltzmann constant
shortwave_absorptivity = 0.7   # ξk
longwave_emissivity = 0.95     # εp
epsilon_p = longwave_emissivity

# Angular factors Fi per SOLWEIG (box-shaped human)
# Horizontal (top, bottom): 0.06
# Vertical (front, back, left, right): 0.22
angular_factors = {
    'front': 0.22,
    'back': 0.22,
    'left': 0.22,
    'right': 0.22,
    'in': 0.06,
    'out': 0.06,
}

# -------------------------------
# Preprocess temperature
# -------------------------------
df['T_sensor_K'] = df['Air_Temp'] + 273.15
df['L_corr_term'] = sigma * (df['T_sensor_K'] ** 4)

# -------------------------------
# Calculate weighted radiation components
# -------------------------------
R_total = 0  # Accumulate weighted total radiative flux

for dir in angular_factors:
    F = angular_factors[dir]

    # Column names
    Qs = df[f'Qs_{dir}']
    QL = df[f'QL_{dir}']

    # Shortwave component: ξk * Ki * Fi
    shortwave = shortwave_absorptivity * Qs * F

    # Longwave component: εp * (Li + correction) * Fi
    longwave = longwave_emissivity * (QL + df['L_corr_term']) * F

    R_total += shortwave + longwave

# -------------------------------
# Compute MRT (Tmrt) using Stefan-Boltzmann law
# -------------------------------
# Calculate Tmrt (K and °C) using corrected Sstr (R_total / εp)
df['MRT_K'] = (R_total / (epsilon_p * sigma)) ** 0.25
df['MRT_C'] = df['MRT_K'] - 273.15

# -------------------------------
# Display
# -------------------------------
print(df[['TIMESTAMP', 'Air_Temp', 'MRT_K', 'MRT_C']].head())

# Save the updated DataFrame with the new MRT columns to a new CSV file.
df.to_csv("C:/thesis/climate_bike/bike_updated_MRT.csv", index=False)

             TIMESTAMP  Air_Temp       MRT_K      MRT_C
0  2023-08-23 11:57:35     25.33  298.969581  25.819581
1  2023-08-23 11:57:36     25.29  298.999236  25.849236
2  2023-08-23 11:57:37     25.29  299.036267  25.886267
3  2023-08-23 11:57:38     25.22  298.968500  25.818500
4  2023-08-23 11:57:39     25.13  298.876940  25.726940


In [6]:
# remove columns I don't need to make data bit cleaner
df = df[df.columns.drop(list(df.filter(regex='^E_')))]
df.to_csv("C:/thesis/climate_bike/bike_updated_MRT_clean.csv", index=False)

In [12]:
# Import necessary libraries
import pandas as pd
import numpy as np

# -------------------------------
# 1. Load the dataset
# -------------------------------
datafile = '../data/raw_data/climate_bike/gertjandatafiets.csv'
df = pd.read_csv(datafile)

# -------------------------------
# 2. Inspect the dataset columns
# -------------------------------
print("Columns in the dataset:")
print(df.columns.tolist())

# -------------------------------
# 3. Check for required columns
# -------------------------------
# For calculating MRT we need:
# - Air_Temp: sensor (or air) temperature in °C (to correct long-wave measurements)
# - Short-wave measurements: Qs_in, Qs_out, Qs_left, Qs_right, Qs_front, Qs_back
# - Long-wave measurements: QL_in, QL_out, QL_left, QL_right, QL_front, QL_back

required_cols = ['Air_Temp', 
                 'Qs_in', 'Qs_out', 'Qs_left', 'Qs_right', 'Qs_front', 'Qs_back',
                 'QL_in', 'QL_out', 'QL_left', 'QL_right', 'QL_front', 'QL_back']

missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError("The following required columns are missing from the dataset: " + str(missing_cols))

# -------------------------------
# 4. Set constants and parameters
# -------------------------------
# Stefan-Boltzmann constant in W/m^2/K^4
sigma = 5.67e-8

# Absorptivity factor for short-wave radiation (commonly ~0.7 for human skin/clothing)
absorptivity = 0.7

# -------------------------------
# 5. Prepare the data for MRT calculation
# -------------------------------
# Convert sensor (air) temperature from °C to Kelvin.
df['T_sensor_K'] = df['Air_Temp'] + 273.15

# Compute the blackbody emission term at the sensor temperature.
# According to the emails, the correction for long-wave radiation is:
#    correction = sigma * T_sensor_K^4
df['L_corr_term'] = sigma * (df['T_sensor_K'] ** 4)

# Define the six directions that are measured.
directions = ['in', 'out', 'left', 'right', 'front', 'back']

# For each direction, combine the shortwave and longwave components.
# The effective radiative flux for a given direction is:
#   E_dir = (QL_dir + L_corr_term) + absorptivity * Qs_dir
# where QL_dir is the raw longwave measurement (which may be negative),
# L_corr_term is the blackbody correction,
# and Qs_dir is the short-wave measurement.
for d in directions:
    Qs_col = 'Qs_' + d
    QL_col = 'QL_' + d
    eff_col = 'E_' + d  # effective flux from this direction
    
    df[eff_col] = (df[QL_col] + df['L_corr_term']) + absorptivity * df[Qs_col]

# Compute the average effective radiative flux from the six directions.
eff_cols = ['E_' + d for d in directions]
df['E_avg'] = df[eff_cols].mean(axis=1)

# -------------------------------
# 6. Calculate Mean Radiant Temperature (MRT)
# -------------------------------
# Using the Stefan-Boltzmann law:
#   sigma * MRT^4 = E_avg   =>   MRT = (E_avg / sigma)^(1/4)
df['MRT_K'] = (df['E_avg'] / sigma) ** 0.25

# Convert MRT to °C if desired:
df['MRT_C'] = df['MRT_K'] - 273.15

# -------------------------------
# 7. Display and (optionally) save the results
# -------------------------------
# Show the first few rows with key columns.
print("\nFirst few rows with MRT values (in Kelvin and °C):")
print(df[['TIMESTAMP', 'Air_Temp', 'MRT_K', 'MRT_C']].head())

# # (Optional) Save the updated DataFrame with the new MRT columns to a new CSV file.
# df.to_csv("../data/raw_data/climate_bike/gertjandatafiets_with_MRT.csv", index=False)

Columns in the dataset:
['TIMESTAMP', 'RECORD', 'GPSData', 'Air_Temp', 'humidity', 'Qs_in', 'Qs_out', 'Qs_left', 'Qs_right', 'Qs_front', 'Qs_back', 'QL_in', 'QL_out', 'QL_left', 'QL_right', 'QL_front', 'QL_back', 'TL_inout', 'TL_leftright', 'TL_front', 'TL_back', 'wheelfreq', 'WindDir', 'WS_ms', 'WSDiag', 'PTemp', 'batt_volt', 'GPSData_time', 'latitude', 'longitude']

First few rows with MRT values (in Kelvin and °C):
         TIMESTAMP  Air_Temp       MRT_K      MRT_C
0  8/23/2023 12:57     25.33  298.723749  25.573749
1  8/23/2023 12:57     25.29  298.737445  25.587445
2  8/23/2023 12:57     25.29  298.770103  25.620103
3  8/23/2023 12:57     25.22  298.703257  25.553257
4  8/23/2023 12:57     25.13  298.613531  25.463531
